# ViCare AI · Mở giao diện hỏi đáp

Chọn **Runtime → Run all** (hoặc **Thời gian chạy → Chạy tất cả**). Nếu Google hỏi quyền chạy mã từ GitHub, hãy xem mã nguồn rồi xác nhận. Sau khi chuẩn bị xong, nhập câu hỏi ở giao diện cuối trang. Có thể thử **“Những yếu tố nào ảnh hưởng đến kết quả đo lưu huyết não?”** hoặc **“Thủ đô của nước Úc là gì?”**.

Mỗi giám khảo dùng máy ảo Colab của mình. Có thể chọn GPU miễn phí khi còn hạn mức, hoặc giữ CPU. Lần đầu cần tải mô hình và dựng 11.227 đoạn; riêng bước dựng chỉ mục CPU có thể mất hơn nửa giờ. Khi mở lại trong cùng máy ảo, ứng dụng dùng lại chỉ mục và mô hình. Notebook này chỉ mở giao diện. [Notebook đánh giá đầy đủ](https://colab.research.google.com/github/Vdbinh05/ViCare-AI-Challenge/blob/ffbc841b5ce371c104b7d6ab4bb16d3c7c8f40f8/ViCare_Vuong_Dinh_Binh.ipynb) · [Xem demo đã có output](https://colab.research.google.com/github/Vdbinh05/ViCare-AI-Challenge/blob/ffbc841b5ce371c104b7d6ab4bb16d3c7c8f40f8/ViCare_Linux_CPU_Executed.ipynb#scrollTo=interactive-question)

Demo học thuật. Vui lòng nhập câu hỏi chung, tránh thông tin định danh và yêu cầu chẩn đoán, kê đơn.

In [ ]:
# @title 1. Chuẩn bị ViCare trên máy ảo Google
import json
import os
import socket
import subprocess
import sys
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.request import ProxyHandler, build_opener
from google.colab import output

revision = "ffbc841b5ce371c104b7d6ab4bb16d3c7c8f40f8"
workspace = Path("/content")
source = workspace / ("vicare-web-" + revision[:12])
runtime = workspace / "vicare-runtime"
runtime.mkdir(exist_ok=True)
if not source.exists():
    with tempfile.TemporaryDirectory(dir=workspace) as scratch:
        checkout = Path(scratch) / "source"
        subprocess.run(["git", "init", "-q", checkout], check=True)
        subprocess.run(["git", "-C", checkout, "fetch", "-q", "--depth", "1",
                        "https://github.com/Vdbinh05/ViCare-AI-Challenge.git", revision], check=True)
        subprocess.run(["git", "-C", checkout, "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
        checkout.rename(source)
if (subprocess.check_output(["git", "-C", source, "rev-parse", "HEAD"], text=True).strip() != revision
        or subprocess.check_output(["git", "-C", source, "status", "--porcelain", "--untracked-files=normal"], text=True).strip()):
    raise RuntimeError("Mã nguồn trong phiên đã thay đổi. Mở một runtime mới.")

env = {k:v for k,v in os.environ.items() if not k.startswith(("VICARE_", "HF_", "OLLAMA_", "PIP_", "UV_"))
       and k not in {"HUGGING_FACE_HUB_TOKEN", "PYTHONPATH", "VIRTUAL_ENV", "GRADIO_SHARE"}}
env.update(HF_HOME=str(runtime / "hf"), HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_TELEMETRY="1",
           VICARE_INDEX_DIR=str(runtime / "chroma_db"), VICARE_LLM_MODEL="qwen3:4b-instruct",
           VICARE_OLLAMA_HOST="http://127.0.0.1:11434", VICARE_OLLAMA_TIMEOUT="180",
           VICARE_OLLAMA_MODELS=str(runtime / "models"), GRADIO_ANALYTICS_ENABLED="False",
           ANONYMIZED_TELEMETRY="False", OPENBLAS_NUM_THREADS="1", PYTHONUNBUFFERED="1",
           VICARE_TORCH_THREADS=str(min(4, len(os.sched_getaffinity(0)))))
def run(args, timeout):
    subprocess.run([str(x) for x in args], cwd=source, env=env, check=True, timeout=timeout)

print("Đang cài môi trường của bản nộp…", flush=True)
run([sys.executable, "-m", "benchmarks.colab_runtime", "install", "--work", runtime], 3600)
python = runtime / "venv/bin/python"
print("Đang chuẩn bị chỉ mục E5 và Qwen3 4B. Lần đầu có thể mất hơn nửa giờ.", flush=True)
with ThreadPoolExecutor(max_workers=2) as pool:
    indexing = pool.submit(run, [python, "-m", "src.retrieval", "build"], 7200)
    model = pool.submit(run, [python, "-m", "benchmarks.colab_runtime", "ollama", "--work", runtime], 3600)
    indexing.result()
    model.result()
print("Chỉ mục và mô hình đã sẵn sàng.", flush=True)

In [ ]:
# @title 2. Hỏi ViCare trong trình duyệt
if "vicare_web" not in globals() or vicare_web.poll() is not None:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        port = s.getsockname()[1]
    log_path = runtime / "gradio-web.log"
    with log_path.open("wb") as log:
        vicare_web = subprocess.Popen([str(python), "-u", "app.py", "--port", str(port)],
                                      cwd=source, env=env, stdout=log, stderr=subprocess.STDOUT)
http = build_opener(ProxyHandler({}))
deadline = time.monotonic() + 180
while time.monotonic() < deadline:
    if vicare_web.poll() is not None:
        raise RuntimeError("Giao diện chưa mở được. Xem nhật ký tại " + str(log_path))
    try:
        with http.open(f"http://127.0.0.1:{port}/config", timeout=3) as response:
            config = json.load(response)
        if any(component.get("props", {}).get("elem_id") == "question"
               for component in config.get("components", [])):
            break
    except (OSError, ValueError):
        pass
    time.sleep(1)
else:
    raise TimeoutError("Giao diện chưa sẵn sàng. Xem " + str(log_path))
output.serve_kernel_port_as_iframe(port, width="100%", height="900")